# 4.2 How accurate are the extracted research claims?

The 2,673 research claims were extracted from 41 sources by an LLM in June 2026 (the claims pipeline, Phase 4). Their **quotes** were validated against the source text automatically; their **labels** were not (`review_status` is still "pending" for all of them). This notebook measures them on a random sample.

**Sample:** `evaluation/claim_label_audit.csv`, 50 claims drawn at random from all 2,673 (seed 2026, `scripts/analysis/make_claim_label_set.py`). For each, compared with its verbatim quote:
- `claim_correct`: the claim says what the quote says, and stands on its own;
- `impact_type_correct`, `direction_correct`, `entity_correct`: the three labels.

**Who judged:** Claude drafted the judgments with a note on every 0, and you accepted them (CHANGELOG 031).

**Scope:** these are the June 2026 claims. Claims added by Phase 8 get their own sample, in the last section (ROADMAP 8.4).

In [1]:
import _setup
_setup.setup()

In [2]:
import pandas as pd
from scipy.stats import binomtest

CHECKS = ["claim_correct", "impact_type_correct", "direction_correct", "entity_correct"]
audit = pd.read_csv("evaluation/claim_label_audit.csv")
audit["all_four"] = audit[CHECKS].all(axis=1).astype(int)


def rate(column, frame=audit):
    k, n = int(frame[column].sum()), len(frame)
    interval = binomtest(k, n).proportion_ci(method="wilson")  # Wilson: sound at n = 50 and rates near 1
    return {"right": k, "of": n, "rate": round(k / n, 2), "95% interval": f"{interval.low:.2f}-{interval.high:.2f}"}


def rates(frame):
    return pd.DataFrame({column: rate(column, frame) for column in CHECKS + ["all_four"]}).T


rates(audit)

,right,of,rate,95% interval
claim_correct,40,50,0.8,0.67-0.89
impact_type_correct,41,50,0.82,0.69-0.90
direction_correct,38,50,0.76,0.63-0.86
entity_correct,42,50,0.84,0.71-0.92
all_four,20,50,0.4,0.28-0.54


## What goes wrong

Every claim with at least one 0, with the note that explains it.

In [3]:
wrong = audit[audit.all_four == 0]
pd.set_option("display.max_colwidth", 120)
wrong[["claim_text", "impact_type_clean", "impact_direction_clean"] + CHECKS + ["note"]].reset_index(drop=True)

,claim_text,impact_type_clean,impact_direction_clean,claim_correct,impact_type_correct,direction_correct,entity_correct,note
0,The AI tool bolstered workers’ daily performance and reduced the time they needed for learning activities.,productivity_effect,negative,1,1,0,1,direction should be positive
1,AI adoption leads to higher sales.,productivity_effect,positive,0,1,1,0,quote '31% higher sales' alone cannot support a causal claim; firms are not an industry
2,"Contracting gives companies access to skills through contractors, freelancers, or temporary workers.",skill_change,positive,1,0,1,1,not about AI; not a skill change
3,The future of good jobs depends in large part on our ability to harness new technology platforms such as artificial ...,job_creation,positive,1,1,0,1,"conditional statement: uncertain, not positive"
4,Responses reveal gendered scoring that aligns with occupational segregation.,automation_exposure,uncertain,0,0,1,0,"no context: whose responses? about bias, not automation"
5,"The highest ranked tasks at work were writing (38 percent), administrative tasks (27 percent), and interpreting/tran...",task_transformation,positive,1,1,0,1,"describes usage, no direction"
6,The current generation of AI technologies is predominantly task replacing and generates only modest productivity gains.,productivity_effect,mixed,1,1,1,0,"entity should be tasks or firms, not 'non-AI hiring'"
7,Usage splits nearly evenly between automation (43%) and augmentation (57%) of human capabilities.,automation_risk,mixed,1,1,1,0,"entity is AI usage, not 'human capabilities'"
8,Employment growth for young workers has been stagnant.,employment_effect,negative,0,1,1,1,'stagnant' overstates 'some leveling off relative to other age groups'
9,Automation can easily complete these tasks and is continually adapting to perform more complicated duties.,automation_risk,negative,0,1,1,1,'these tasks' unresolved in the claim


## Which of these reach students?

In [4]:
shown = pd.read_csv("data/research/career_research_claims.csv", dtype=str)
in_app = audit[audit.claim_id.isin(shown.claim_id)]
print(f"claims the app shows: {shown.claim_id.nunique()} distinct, across {shown.onetsoc_code.nunique()} careers")
print(f"audited claims among them: {len(in_app)} of {len(audit)}; claim text right: {in_app.claim_correct.sum()} of {len(in_app)}")
in_app.loc[in_app.claim_correct == 0, ["claim_text", "note"]].reset_index(drop=True)

claims the app shows: 402 distinct, across 740 careers
audited claims among them: 8 of 50; claim text right: 7 of 8


,claim_text,note
0,The companies that are achieving the largest productivity gains are redesigning how they operate.,claim says something the quote does not


## The claims added in October 2026

The first monthly update (CHANGELOG 057) added 610 claims from 3 papers: a review of AI and jobs (arXiv 2509.15265), a study of generative AI and firm values (NBER w31222) and a theory paper, "The AI Layoff Trap" (arXiv 2603.20617). **Sample:** `evaluation/claim_label_audit_2026_10.csv`, 50 drawn at random from the 610 (seed 2026), judged the same way: Claude drafted, you accepted (CHANGELOG 060).

In [5]:
new = pd.read_csv("evaluation/claim_label_audit_2026_10.csv")
new["all_four"] = new[CHECKS].all(axis=1).astype(int)
pd.concat({"June 2026 claims": rates(audit), "October 2026 claims": rates(new)}, axis=1)

June 2026 claims                         \
                               right  of  rate 95% interval   
claim_correct                     40  50   0.8    0.67-0.89   
impact_type_correct               41  50  0.82    0.69-0.90   
direction_correct                 38  50  0.76    0.63-0.86   
entity_correct                    42  50  0.84    0.71-0.92   
all_four                          20  50   0.4    0.28-0.54   

                    October 2026 claims                         
                                  right  of  rate 95% interval  
claim_correct                        31  50  0.62    0.48-0.74  
impact_type_correct                  38  50  0.76    0.63-0.86  
direction_correct                    36  50  0.72    0.58-0.83  
entity_correct                       43  50  0.86    0.74-0.93  
all_four                             18  50  0.36    0.24-0.50

In [6]:
in_app = new[new.claim_id.isin(shown.claim_id)]
print(f"audited new claims the app shows: {len(in_app)} of {len(new)}; claim text right: {in_app.claim_correct.sum()} of {len(in_app)}")
new.loc[new.claim_correct == 0, ["claim_text", "note"]].reset_index(drop=True)

audited new claims the app shows: 7 of 50; claim text right: 4 of 7


,claim_text,note
0,The model proved too restrictive to explain declining labor shares observed across advanced economies since the 1980s.,"'the model' unresolved; about an economic model's limits, not AI: type uncertainty_or_limitation, no direction"
1,Regular users spent 3.6 fewer hours per week on email—a 31% reduction.,'regular users' of what? the AI tool isn't named
2,The effects of Generative AI exposure on labor demand and profitability are more than twice as large when derived fr...,"'when derived from core tasks' is not in the quote; profitability is firm-level, not the general labor market"
3,AI adoption leads to a reallocation of time from project management and coordination activities towards core technic...,"whose work? (software developers in the source); without it, it reads as true of any job"
4,Significant job reductions in these key occupational clusters could disrupt existing pathways for occupational mobil...,'these key occupational clusters' unresolved
5,AI can bridge disciplines and improve creative discovery.,'bridge disciplines' is not in the quote
6,"The new technology redistributes income in the economy, which may or may not be desirable from a welfare perspective.",'the new technology' unresolved and the condition dropped; 'may or may not be desirable': uncertain
7,"The resolution would be Pyrrhic, with retained workers earning barely more than the machines that would replace them.",'the resolution' unresolved
8,AI improves both speed and quality across all tasks.,'all tasks' are one study's tasks; the claim reads as universal
9,The impact on inequality is inconclusive.,impact of what? AI isn't named


## Findings

1. **Claim texts are mostly right overall (4 in 5), but less so among the claims the app shows**: 9 of the 14 audited claims that reach students. The failures there are claims that **don't stand on their own** ("The sector could experience…", "…complete these tasks…") or say more than the quote. Showing the quote doesn't fix a missing context, because the quote lacks it too.
   - A possible fix: ask the relevance check (4.1c) to also reject claims that don't stand on their own. That is a prompt change and a full re-run (about $1.40), so it's yours to decide; noted in ROADMAP 4.2.
2. **The labels are weaker**: only about 2 in 5 claims are right on all four checks, and direction is the least reliable (e.g. "negative" for a quote about workers' performance improving). **So the app shows no impact type or direction**, only the claim, the quote and the source.
3. **A few claims aren't about AI** although they passed the extraction's AI filter (3 of 50), and one of them, about contracting, is shown by the app.
4. **The October claims stand on their own less often**: 31 of 50 claim texts are right (0.62), against 40 of 50 among the June claims, and the intervals barely overlap. Most failures are references the claim doesn't resolve ("these reinstatement effects", "the resolution"), terms of the theory paper's model ("Both groups lose at αNE"), and claims that drop whose work they describe ("AI reallocates time from project management…", about software developers), which then read as true of any job. The labels are about as good as in June. This is more evidence for the fix in item 1 (ROADMAP 4.2).

**Limits:** 50 claims give wide intervals (see the table), and 14 shown claims are too few for a reliable rate; the judgments are Claude's, accepted by you.